In [1]:
!pip -q install \
google-generativeai \
sentence-transformers \
faiss-cpu \
PyPDF2 \
numpy

In [2]:
!pip install -q wandb

In [3]:
import wandb

wandb.login()
wandb.init(
    project="UAS-RAG-Gemini",
    name="Eksperimen-RAG",
)
table = wandb.Table(columns=[
    "Pertanyaan",
    "Jawaban Tanpa RAG",
    "Jawaban Dengan RAG",
    "Status"
])

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: salwahafizhahyus (salwahafizhahyus-stikomelrahma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
import os
import numpy as np
import faiss
import google.generativeai as genai

from google.colab import files
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer

print("✅ Semua library berhasil diimport")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Semua library berhasil diimport


In [7]:
import pandas as pd

results = []

In [8]:
def ask_without_rag(question):
    response = model.generate_content(question)
    return response.text

In [9]:
GOOGLE_API_KEY = "AQ.Ab8RN6LhK1zSGlXT-....."

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

print("✅ Gemini siap digunakan")

✅ Gemini siap digunakan


In [10]:
uploaded = files.upload()

Saving Data Science_Minggu 1 & 2_Salwa Hafizhah Y_2411070055.pdf to Data Science_Minggu 1 & 2_Salwa Hafizhah Y_2411070055 (6).pdf


In [11]:
pdf_name = list(uploaded.keys())[0]

reader = PdfReader(pdf_name)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("Jumlah karakter:", len(text))

Jumlah karakter: 12444


In [12]:
chunk_size = 500
overlap = 100

chunks = []

start = 0

while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start += chunk_size - overlap

print("Jumlah chunk:", len(chunks))

Jumlah chunk: 32


In [13]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("✅ FAISS berhasil dibuat")
print("Jumlah vector:", index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ FAISS berhasil dibuat
Jumlah vector: 32


In [14]:
def retrieve_context(query, top_k=3):
    # Ubah pertanyaan menjadi embedding
    query_embedding = embedding_model.encode([query])

    # Cari chunk yang paling mirip
    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    # Ambil isi chunk
    retrieved_chunks = [chunks[i] for i in indices[0]]

    return "\n\n".join(retrieved_chunks)

In [15]:
def ask_with_rag(question):
    context = retrieve_context(question)

    prompt = f"""
Kamu hanya boleh menjawab berdasarkan context berikut.

Context:
{context}

Pertanyaan:
{question}

Jika jawaban tidak ditemukan pada context, jawab:

"Informasi tersebut tidak ditemukan pada dokumen."
"""

    response = model.generate_content(prompt)

    return response.text

In [16]:
def log_to_wandb(question):

    no_rag = ask_without_rag(question)

    with_rag = ask_with_rag(question)

    status = "✅ Ditemukan"

    if "tidak ditemukan" in with_rag.lower():
        status = "❌ Tidak ditemukan"

    table.add_data(
        question,
        no_rag,
        with_rag,
        status
    )

    wandb.log({
        "Perbandingan RAG": table
    })

    return no_rag, with_rag

In [19]:
print("=" * 60)
print("📚 RAG Chatbot dengan Gemini")
print("Setiap pertanyaan otomatis disimpan ke W&B")
print("Ketik 'exit' untuk keluar")
print("=" * 60)

while True:

    question = input("\nPertanyaan : ")

    if question.lower() == "exit":
        print("Terima kasih 👋")
        break

    no_rag, rag = log_to_wandb(question)

    print("\n" + "=" * 60)
    print("📌 TANPA RAG")
    print("=" * 60)
    print(no_rag)

    print("\n" + "=" * 60)
    print("📌 DENGAN RAG")
    print("=" * 60)
    print(rag)

📚 RAG Chatbot dengan Gemini
Setiap pertanyaan otomatis disimpan ke W&B
Ketik 'exit' untuk keluar

Pertanyaan : berapa kah probalitas dari data siswa ?


TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 19.700622366s.